In [ ]:
# Run in a fresh runtime, and use %pip (targets the notebook kernel)
%pip uninstall -y torch torchvision torchaudio xformers pillow
%pip cache purge -q

%pip install --no-cache-dir \
  torch torchvision pillow \
  --index-url https://download.pytorch.org/whl/cu126

%pip install --no-cache-dir -U xformers --index-url https://download.pytorch.org/whl/cu126


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive/')
current_file_path = '/content/drive/MyDrive/ve/variance_analysis'

os.chdir(current_file_path)

%pip install -e .

!mkdir -p /usr/share/fonts/truetype/helvetica
!ttx -y 0 -o /usr/share/fonts/truetype/helvetica/Helvetica-Regular.ttf fonts/Helvetica.ttc
!ttx -y 1 -o /usr/share/fonts/truetype/helvetica/Helvetica-Oblique.ttf fonts/Helvetica.ttc
!ttx -y 2 -o /usr/share/fonts/truetype/helvetica/Helvetica-Bold.ttf fonts/Helvetica.ttc
!ttx -y 3 -o /usr/share/fonts/truetype/helvetica/Helvetica-BoldOblique.ttf fonts/Helvetica.ttc
!fc-cache -f

from pathlib import Path
current_file_path = Path(current_file_path)

# Visualize Model Neurons

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
import matplotlib.font_manager as fm
import shutil

!mkdir -p ~/.fonts
!cp fonts/Helvetica.ttc ~/.fonts/

cache_dir = os.path.join(os.path.expanduser("~"), ".cache", "matplotlib")
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)

fm._load_fontmanager(try_read_cache=False)
plt.rcParams['font.family'] = 'Helvetica'

## Extract panorama data

In [ ]:
import numpy as np

model_name = 'predictive_coder'  # predictive_coder | resnet50 | vit_b16 | resnet50_untrained | vgg16

# Environment constants
ENV_X_MIN = 120
ENV_X_MAX = 136
ENV_Y_MIN = 197
ENV_Y_MAX = 213

# Grid dimensions
N_POS_X = ENV_X_MAX - ENV_X_MIN + 1  # 17
N_POS_Y = ENV_Y_MAX - ENV_Y_MIN + 1  # 17

N_HD_PANORAMA = 90

print(f"Grid: {N_POS_X} x {N_POS_Y} locations")

# Variance Explained Analysis

Now let's see how much variance can be attributed to each of the possible stimulus dimensions for each neuron.

**Data sources:**
- `panorama-90yaw`: 90 HD samples per location (4° resolution) at 0° pitch → used for place cell classification
- `panorama-90yaw-pitch`: 90 HD × 4 pitch = 360 samples per location → used for EV calculation

**Visit Mode:**
- `"pitch"`: Each (xy, hd) bin has 4 visits from different pitch angles [-3°, -1°, +1°, +3°].
  Folds are assigned interleaved: fold0 = [-3°, +1°], fold1 = [-1°, +3°]

In [ ]:
# predictive code latents are pre-prediction/post-encoding ('pre_pred'), or post-prediction/pre-decoding residual stream ('post_pred')
# resnet can take layer1, layer3, layer4, or avgpool
# vit can take layer_{3, 6, 9, 12}
# vgg16 can take features.9, features.16, features.30, or classifier.3

from src.ann.get_activations import load_model

layer_name = 'pre_pred'

config = {
    'model_name': model_name,
    'output_layer': layer_name,
    'env_bounds': [ENV_X_MIN, ENV_X_MAX, ENV_Y_MIN, ENV_Y_MAX],
    'n_rotations': N_HD_PANORAMA  # Using 90 HD samples from panorama
}

model, preprocess = load_model(model_name, base_path=current_file_path, output_layer=layer_name)

In [ ]:
# Panorama data configuration
# --------------------------
# panorama-90yaw: 90 HD samples at 0° pitch (for place cell classification)
# panorama-90yaw-pitch: 90 HD × 4 pitch samples (for EV with pitch visits)

# Pitch configuration
N_PITCHES = 4
PITCHES = [-3, -1, 1, 3]  # degrees

# Interleaved fold assignment to balance positive/negative pitches
# fold0 gets pitches at indices [0, 2] -> [-3°, +1°]
# fold1 gets pitches at indices [1, 3] -> [-1°, +3°]
FOLD0_PITCH_INDICES = [0, 2]
FOLD1_PITCH_INDICES = [1, 3]

device = 'cuda'

In [ ]:
# Load panorama-90yaw data (clean, 0° pitch) for place cell classification
import glob
from tqdm.auto import tqdm

def load_panorama_images(panorama_dir, with_pitch=False):
    """
    Load panorama images from npz files.

    Args:
        panorama_dir: Path to directory containing loc_*.npz files
        with_pitch: If True, expects 360 images per location (90 HD * 4 pitch)
                   If False, expects 90 images per location (90 HD only)

    Returns:
        images: (n_locs, n_images_per_loc, H, W, C) array
        locations: (n_locs, 2) array of (x, y) coordinates
    """
    npz_files = sorted(glob.glob(str(panorama_dir / 'loc_*.npz')))
    print(f"Found {len(npz_files)} location files in {panorama_dir}")

    images_list = []
    locations_list = []

    for npz_file in tqdm(npz_files, desc="Loading panorama"):
        data = np.load(npz_file)
        imgs = data['rgb']  # (n_images, H, W, C) or similar

        # Parse location from filename: loc_+XXX.X_+YYY.Y.npz
        fname = Path(npz_file).stem
        parts = fname.replace('loc_', '').split('_')
        x = float(parts[0])
        y = float(parts[1])

        images_list.append(imgs)
        locations_list.append([x, y])

    images = np.stack(images_list, axis=0)  # (n_locs, n_images, H, W, C)
    locations = np.array(locations_list)     # (n_locs, 2)

    expected_per_loc = 360 if with_pitch else 90
    print(f"Loaded images shape: {images.shape}")
    print(f"Expected {expected_per_loc} images per location, got {images.shape[1]}")

    return images, locations


# Load clean panorama (90 HD, 0° pitch)
panorama_clean_dir = current_file_path / 'data/minecraft/panorama-90yaw'
panorama_clean_images, panorama_clean_locs = load_panorama_images(panorama_clean_dir, with_pitch=False)

n_locs_clean = panorama_clean_images.shape[0]
print(f"\nClean panorama: {n_locs_clean} locations, {panorama_clean_images.shape[1]} HD samples each")

# Show an example image
plt.imshow(panorama_clean_images[0, 0])
plt.title(f"Location {panorama_clean_locs[0]}, HD=0")
plt.show()

In [ ]:
import torch

def get_panorama_activations(model, preprocess_fn, panorama_images, model_name,
                              device='cuda', batch_size=32, max_units=8000, sample_seed=413):
    """
    Compute model activations for panorama images.

    Args:
        model: PyTorch model
        preprocess_fn: Preprocessing function from load_model (expects numpy arrays)
        panorama_images: (n_locs, n_images_per_loc, H, W, C) array
        model_name: Name of the model (needed to handle predictive_coder specially)
        device: 'cuda' or 'cpu'
        batch_size: Batch size for inference
        max_units: Maximum number of units to keep (subsamples if exceeded)
        sample_seed: Random seed for unit subsampling

    Returns:
        activations: (n_locs, n_images_per_loc, n_neurons) array
    """
    device = torch.device(device) if isinstance(device, str) else device
    model.to(device).eval()

    n_locs, n_imgs_per_loc = panorama_images.shape[:2]

    # Flatten for batch processing
    flat_images = panorama_images.reshape(-1, *panorama_images.shape[2:])  # (n_total, H, W, C)
    n_total = flat_images.shape[0]

    # Check if predictive coder (needs sequence dimension)
    is_predictive = model_name.startswith("predictive_coder")
    seq_len = 10 if is_predictive else None

    selected_idx = None
    activations_list = []

    with torch.no_grad():
        for i in tqdm(range(0, n_total, batch_size), desc="Computing activations"):
            batch_np = flat_images[i:i+batch_size]  # (B, H, W, C)

            # For predictive coder, add sequence dimension
            if is_predictive:
                batch_np = np.repeat(batch_np[:, None, ...], seq_len, axis=1)  # (B, T, H, W, C)

            # Apply preprocessing and move to device
            x = preprocess_fn(batch_np).to(device)

            # Forward pass
            out = model(x)

            # Handle different output formats
            if isinstance(out, dict):  # torchvision feature extractor
                out = next(iter(out.values()))
            if is_predictive:
                out = out[:, -1]  # Take last timestep

            # Flatten spatial dimensions
            out = out.flatten(1)

            # Subsample units if needed
            if out.shape[1] > max_units:
                if selected_idx is None:
                    g = torch.Generator().manual_seed(sample_seed)  # CPU generator, so the same units are picked on any device
                    selected_idx = torch.randperm(out.shape[1], generator=g)[:max_units].to(out.device)
                out = out.index_select(1, selected_idx)

            activations_list.append(out.cpu().numpy())

    activations_flat = np.concatenate(activations_list, axis=0)  # (n_total, n_neurons)
    activations = activations_flat.reshape(n_locs, n_imgs_per_loc, -1)  # (n_locs, n_imgs, n_neurons)

    return activations


# Compute or load clean activations
clean_acts_path = current_file_path / f'data/ann_activations/{model_name}_{layer_name}_panorama_clean/acts.npz'

if clean_acts_path.exists():
    print(f"Loading cached clean activations from {clean_acts_path}")
    clean_acts_data = np.load(clean_acts_path)
    clean_acts = clean_acts_data['activations']
else:
    print("Computing clean activations from panorama-90yaw...")
    clean_acts = get_panorama_activations(
        model, preprocess, panorama_clean_images,
        model_name=model_name,
        device=device, batch_size=64
    )

    # Cache for later
    clean_acts_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(clean_acts_path, activations=clean_acts, locations=panorama_clean_locs)
    print(f"Saved clean activations to {clean_acts_path}")

print(f"Clean activations shape: {clean_acts.shape}")
# Shape: (n_locs, n_hd, n_neurons) = (289, 90, n_neurons)

n_locs, n_hd, n_neurons = clean_acts.shape
print(f"  {n_locs} locations, {n_hd} HD samples, {n_neurons} neurons")

In [ ]:
from src.ann.classify_neurons import score_neurons, classify_neurons

# For score_neurons, we need activations in the expected format
# Reshape clean_acts from (n_locs, n_hd, n_neurons) to (n_locs * n_hd, n_neurons)
#clean_acts_flat = clean_acts.reshape(-1, n_neurons)

# Save in expected format temporarily
temp_acts_path = current_file_path / f'data/ann_activations/{model_name}_{layer_name}_panorama_clean/acts_flat.npz'
#np.savez(temp_acts_path, activations=clean_acts_flat)

# Update config for panorama data
config_classify = {
    'model_name': model_name,
    'output_layer': layer_name,
    'env_bounds': [ENV_X_MIN, ENV_X_MAX, ENV_Y_MIN, ENV_Y_MAX],
    'n_rotations': N_HD_PANORAMA
}

neuron_scores_fpath, sampled_path = score_neurons(
    [temp_acts_path],
    config_classify,
    base_dir=current_file_path,
    sample_partial=False,
    pixel_min_threshold=1,
    use_old_place_method=False,
    n_visits=1,
)

neuron_score_charts = np.load(neuron_scores_fpath, allow_pickle=True)
classifications = classify_neurons(neuron_score_charts)
print(classifications.keys())

In [ ]:
from src.ann.fig import plot_cell_type_pie

fig = plot_cell_type_pie(classifications)
fig.savefig(current_file_path / f'figs/{model_name}_{layer_name}_breakdown.pdf')
plt.show()

In [ ]:
def get_splits_deterministic(visits, fold0_indices, fold1_indices):
    """
    Split visits into folds using deterministic index assignment.

    This is used for pitch-based visits where we want balanced pitch assignment:
    - fold0 gets pitches at fold0_indices (e.g., [-3°, +1°])
    - fold1 gets pitches at fold1_indices (e.g., [-1°, +3°])

    Args:
        visits: (s1_bins, s2_bins, n_visits, n_neurons) array
                For pitch mode: (n_locs, n_hd, n_pitches, n_neurons)
        fold0_indices: list of visit indices for fold0
        fold1_indices: list of visit indices for fold1

    Returns:
        fold0_mean, fold1_mean: (s1_bins, s2_bins, n_neurons) arrays
        fold0_occ, fold1_occ: (s1_bins, s2_bins) occupancy arrays
    """
    s1_bins, s2_bins, n_visits, n_neurons = visits.shape

    # Extract visits for each fold
    fold0_visits = visits[:, :, fold0_indices, :]  # (s1, s2, n_fold0, neurons)
    fold1_visits = visits[:, :, fold1_indices, :]  # (s1, s2, n_fold1, neurons)

    # Compute means across visits dimension
    fold0_mean = np.nanmean(fold0_visits, axis=2)  # (s1, s2, neurons)
    fold1_mean = np.nanmean(fold1_visits, axis=2)  # (s1, s2, neurons)

    # Occupancy is uniform since we always have the same number of visits per bin
    fold0_occ = np.full((s1_bins, s2_bins), len(fold0_indices), dtype=float)
    fold1_occ = np.full((s1_bins, s2_bins), len(fold1_indices), dtype=float)

    return fold0_mean, fold1_mean, fold0_occ, fold1_occ


print(f"Pitch visit mode configuration:")
print(f"  Pitches: {PITCHES}")
print(f"  Fold 0 indices: {FOLD0_PITCH_INDICES} -> pitches {[PITCHES[i] for i in FOLD0_PITCH_INDICES]}")
print(f"  Fold 1 indices: {FOLD1_PITCH_INDICES} -> pitches {[PITCHES[i] for i in FOLD1_PITCH_INDICES]}")

## Load panorama-pitch data for EV calculation

In [ ]:
# Load the pitch-varied panorama data
panorama_pitch_dir = current_file_path / 'data/minecraft/panorama-90yaw-pitch'
panorama_pitch_images, panorama_pitch_locs = load_panorama_images(panorama_pitch_dir, with_pitch=True)

n_locs_pitch = panorama_pitch_images.shape[0]
print(f"\nPitch panorama: {n_locs_pitch} locations")
print(f"Images per location: {panorama_pitch_images.shape[1]} (should be {N_HD_PANORAMA * N_PITCHES} = {N_HD_PANORAMA * N_PITCHES})")

# Verify locations match
assert n_locs_pitch == n_locs, f"Location count mismatch: pitch={n_locs_pitch}, clean={n_locs}"

In [ ]:
pitch_acts_path = current_file_path / f'data/ann_activations/{model_name}_{layer_name}_panorama_pitch/acts.npz'

if pitch_acts_path.exists():
    print(f"Loading cached pitch activations from {pitch_acts_path}")
    pitch_acts = np.load(pitch_acts_path)['activations']
else:
    print("Computing pitch activations (this may take a while)...")
    pitch_acts = get_panorama_activations(
        model, preprocess, panorama_pitch_images,
        model_name=model_name,
        device=device, batch_size=64
    )

    # Cache for later
    pitch_acts_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(pitch_acts_path, activations=pitch_acts, locations=panorama_pitch_locs)
    print(f"Saved pitch activations to {pitch_acts_path}")

print(f"Pitch activations shape: {pitch_acts.shape}")

## Compute Explained Variance

In [ ]:
from tqdm import tqdm
from src.explained_variance import nan_aware_gaussian_filter, masked_zoom, quantify_variance_corr

place_cells = np.asarray(classifications['exclusive_place_cells_indices'])
is_place = np.isin(np.arange(n_neurons), place_cells)
print(f"Found {len(place_cells)} place cells out of {n_neurons} neurons")

sigma = (1, 1)

pitch_acts_4d = pitch_acts.reshape(n_locs, N_HD_PANORAMA, N_PITCHES, n_neurons)
print(f"Pitch activations reshaped: {pitch_acts_4d.shape}")
print(f"  = (n_locs={n_locs}, n_hd={N_HD_PANORAMA}, n_pitch={N_PITCHES}, n_neurons={n_neurons})")

fold0_mean, fold1_mean, fold0_occ, fold1_occ = get_splits_deterministic(
    pitch_acts_4d,
    FOLD0_PITCH_INDICES,
    FOLD1_PITCH_INDICES
)

fold0 = fold0_mean
fold1 = fold1_mean
occ = fold0_occ + fold1_occ

zoom_factor = (1, N_HD_PANORAMA // N_POS_X)

print(f"fold0 shape: {fold0.shape}")
print(f"fold1 shape: {fold1.shape}")
print(f"occ shape: {occ.shape}")
print(f"zoom_factor: {zoom_factor}")


In [ ]:
from joblib import Parallel, delayed

n_using = min(1000, n_neurons)
neurons = list(range(n_neurons))
np.random.seed(42)
np.random.shuffle(neurons)

def process_neuron_pitch(n, fold0, fold1, occ, zoom_factor, sigma):
    f0 = nan_aware_gaussian_filter(fold0[:, :, n], sigma=sigma, circular_axes=(1,))
    f1 = nan_aware_gaussian_filter(fold1[:, :, n], sigma=sigma, circular_axes=(1,))
    f0 = masked_zoom(f0, zoom_factor=zoom_factor)
    f1 = masked_zoom(f1, zoom_factor=zoom_factor)
    occ_z = masked_zoom(occ, zoom_factor=zoom_factor)
    ev_xy = 1 - quantify_variance_corr(f0, f1, occ_z, axis=0)[0]
    ev_hd = 1 - quantify_variance_corr(f0, f1, occ_z, axis=1)[0]
    return ev_xy, ev_hd

results = Parallel(n_jobs=-1)(
    delayed(process_neuron_pitch)(neurons[i], fold0, fold1, occ, zoom_factor, sigma)
    for i in tqdm(range(n_using))
)

s1_var = np.array([r[0] for r in results])
s2_var = np.array([r[1] for r in results])
is_place_trunc = np.array([is_place[neurons[i]] for i in range(n_using)])

print(f"Computed EV for {n_using} neurons")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

n_sample_other = 200
n_sample_place = 100
rng = np.random.default_rng(42)

other_idx = np.where(~is_place_trunc)[0]
place_idx = np.where(is_place_trunc)[0]

if len(other_idx) > n_sample_other:
    other_idx = rng.choice(other_idx, n_sample_other, replace=False)
if len(place_idx) > n_sample_place:
    place_idx = rng.choice(place_idx, n_sample_place, replace=False)

plt.scatter(s1_var[other_idx], s2_var[other_idx], s=30,
            label=f"Other neurons (n={len(other_idx)})")
plt.scatter(s1_var[place_idx], s2_var[place_idx], color="red", s=30,
            label=f"Exclusive place cells (n={len(place_idx)})")

vals = np.concatenate([s1_var, s2_var])
vmin, vmax = 0, 1 #np.nanmin(vals), np.nanmax(vals)
plt.plot([vmin, vmax], [vmin, vmax], "--", lw=1, color="k", alpha=0.6)

plt.gca().set_aspect("equal", adjustable="box")
plt.xlabel("EV (XY)")
plt.ylabel("EV (HD)")
plt.ylim(0.0, 1.0)
plt.xlim(0.0, 1.0)
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig(current_file_path / f'figs/{model_name}_{layer_name}_ev_pitch.pdf')
plt.show()

x_all = np.asarray(s1_var, dtype=float).ravel()
y_all = np.asarray(s2_var, dtype=float).ravel()
valid = np.isfinite(x_all) & np.isfinite(y_all)
xv, yv = x_all[valid], y_all[valid]

xy_dominant = np.sum(xv > yv) / len(xv)
hd_dominant = np.sum(yv > xv) / len(yv)
ratio = yv / (xv + yv + 1e-8)
print(f"\nSummary for pitch visit mode (all {len(xv)} neurons):")
print(f"  XY-dominant: {xy_dominant:.1%}")
print(f"  HD-dominant: {hd_dominant:.1%}")
print(f"  Mean HD/(XY+HD): {np.mean(ratio):.3f}")
print(f"  Median HD/(XY+HD): {np.median(ratio):.3f}")


## Loop for all models and layers

In [ ]:
from pathlib import Path
import numpy as np
from tqdm.auto import tqdm
from joblib import Parallel, delayed

from src.ann.get_activations import load_model
from src.ann.classify_neurons import score_neurons, classify_neurons
from src.explained_variance import nan_aware_gaussian_filter, masked_zoom, quantify_variance_corr

models_and_layers = {
    "resnet50": ["layer1", "layer3", "layer4", "avgpool"],
    "resnet50_untrained": ["avgpool"],
    "vgg16": ["features.9"],
    "predictive_coder": ["conv1", "down1", "pre_pred", "post_pred"],
}

def process_neuron_loop(n, fold0, fold1, occ, zoom_factor, sigma):
    f0 = nan_aware_gaussian_filter(fold0[:, :, n], sigma=sigma, circular_axes=(1,))
    f1 = nan_aware_gaussian_filter(fold1[:, :, n], sigma=sigma, circular_axes=(1,))
    f0 = masked_zoom(f0, zoom_factor=zoom_factor)
    f1 = masked_zoom(f1, zoom_factor=zoom_factor)
    occ_z = masked_zoom(occ, zoom_factor=zoom_factor)
    ev_xy = 1 - quantify_variance_corr(f0, f1, occ_z, axis=0)[0]
    ev_hd = 1 - quantify_variance_corr(f0, f1, occ_z, axis=1)[0]
    return ev_xy, ev_hd

for model_name, layer_names in models_and_layers.items():
    for layer_name in layer_names:
        print(f"\n{'='*60}")
        print(f"=== {model_name} / {layer_name} ===")
        print(f"{'='*60}")

        model_curr, preprocess_curr = load_model(
            model_name, base_path=current_file_path, output_layer=layer_name
        )

        clean_acts_path = current_file_path / f'data/ann_activations/{model_name}_{layer_name}_panorama_clean/acts.npz'

        if clean_acts_path.exists():
            print(f"  Loading cached clean activations")
            clean_acts_curr = np.load(clean_acts_path)['activations']
        else:
            print(f"  Computing clean activations...")
            clean_acts_curr = get_panorama_activations(
                model_curr, preprocess_curr, panorama_clean_images,
                model_name=model_name, device=device, batch_size=64
            )
            clean_acts_path.parent.mkdir(parents=True, exist_ok=True)
            np.savez_compressed(clean_acts_path, activations=clean_acts_curr, locations=panorama_clean_locs)

        n_locs_curr, n_hd_curr, n_neurons_curr = clean_acts_curr.shape
        print(f"  n_neurons={n_neurons_curr}")

        clean_acts_flat = clean_acts_curr.reshape(-1, n_neurons_curr)
        temp_acts_path = current_file_path / f'data/ann_activations/{model_name}_{layer_name}_panorama_clean/acts_flat.npz'
        np.savez(temp_acts_path, activations=clean_acts_flat)

        config_curr = {
            "model_name": model_name,
            "output_layer": layer_name,
            "env_bounds": [ENV_X_MIN, ENV_X_MAX, ENV_Y_MIN, ENV_Y_MAX],
            "n_rotations": N_HD_PANORAMA,
        }

        neuron_scores_fpath, _ = score_neurons(
            [temp_acts_path], config_curr, base_dir=current_file_path,
            sample_partial=False, pixel_min_threshold=1,
            use_old_place_method=False, n_visits=1,
        )
        neuron_score_charts = np.load(neuron_scores_fpath, allow_pickle=True)
        classifications_curr = classify_neurons(neuron_score_charts)
        place_cells_curr = np.asarray(classifications_curr["exclusive_place_cells_indices"])
        is_place_curr = np.isin(np.arange(n_neurons_curr), place_cells_curr)
        print(f"  {len(place_cells_curr)} place cells")

        pitch_acts_path = current_file_path / f'data/ann_activations/{model_name}_{layer_name}_panorama_pitch/acts.npz'

        if pitch_acts_path.exists():
            print(f"  Loading cached pitch activations")
            pitch_acts_curr = np.load(pitch_acts_path)['activations']
        else:
            print(f"  Computing pitch activations...")
            pitch_acts_curr = get_panorama_activations(
                model_curr, preprocess_curr, panorama_pitch_images,
                model_name=model_name, device=device, batch_size=64
            )
            pitch_acts_path.parent.mkdir(parents=True, exist_ok=True)
            np.savez_compressed(pitch_acts_path, activations=pitch_acts_curr, locations=panorama_pitch_locs)

        sigma = (1, 1)
        pitch_acts_4d = pitch_acts_curr.reshape(n_locs_curr, N_HD_PANORAMA, N_PITCHES, n_neurons_curr)

        fold0_curr, fold1_curr, fold0_occ_curr, fold1_occ_curr = get_splits_deterministic(
            pitch_acts_4d, FOLD0_PITCH_INDICES, FOLD1_PITCH_INDICES
        )
        occ_curr = fold0_occ_curr + fold1_occ_curr
        zoom_factor_curr = (1, N_HD_PANORAMA // N_POS_X)

        results = Parallel(n_jobs=-1)(
            delayed(process_neuron_loop)(n, fold0_curr, fold1_curr, occ_curr, zoom_factor_curr, sigma)
            for n in tqdm(range(n_neurons_curr), desc=f"  EV per neuron")
        )

        ev_xy = np.array([r[0] for r in results])
        ev_hd = np.array([r[1] for r in results])

        ev_out_dir = current_file_path / f"results/ann/{model_name}_{layer_name}"
        ev_out_dir.mkdir(parents=True, exist_ok=True)

        np.savez(
            ev_out_dir / "ev_all_neurons_pitch.npz",
            ev_xy=ev_xy, ev_hd=ev_hd,
            is_place=is_place_curr, n_neurons=n_neurons_curr,
        )
        print(f"  Saved EVs for {n_neurons_curr} neurons")

        valid = np.isfinite(ev_xy) & np.isfinite(ev_hd)
        x, y = ev_xy[valid], ev_hd[valid]
        xy_dominant = np.sum(x > y) / len(x)
        ratio = y / (x + y + 1e-8)
        print(f"  XY-dominant: {xy_dominant:.1%}, Mean HD/(XY+HD): {np.mean(ratio):.3f}")


## Generate scatter plots for all models

In [ ]:
n_sample_other = 250
n_sample_place = 75
rng_vis = np.random.default_rng(42)

for model_name_plot, layer_names_plot in models_and_layers.items():
    for layer_name_plot in layer_names_plot:
        print(f"\n=== Plotting {model_name_plot} / {layer_name_plot} ===")

        ev_path = current_file_path / f"results/ann/{model_name_plot}_{layer_name_plot}/ev_all_neurons_pitch.npz"
        if not ev_path.exists():
            print(f"  Skipping: EV file not found at {ev_path}")
            continue

        data = np.load(ev_path)
        ev_xy, ev_hd = data["ev_xy"], data["ev_hd"]
        is_place_plot = data["is_place"]
        n_neurons_plot = int(data["n_neurons"])
        print(f"  Loaded EVs for {n_neurons_plot} neurons ({is_place_plot.sum()} place cells)")

        other_idx = np.where(~is_place_plot)[0]
        place_idx = np.where(is_place_plot)[0]

        if len(other_idx) > n_sample_other:
            other_idx = rng_vis.choice(other_idx, n_sample_other, replace=False)
        if len(place_idx) > n_sample_place:
            place_idx = rng_vis.choice(place_idx, n_sample_place, replace=False)

        plt.scatter(ev_xy[other_idx], ev_hd[other_idx], s=30,
                    label=f"Other neurons (n={len(other_idx)})")
        plt.scatter(ev_xy[place_idx], ev_hd[place_idx], color="red", s=30,
                    label=f"Exclusive place cells (n={len(place_idx)})")

        vals = np.concatenate([ev_xy.ravel(), ev_hd.ravel()])
        vmin, vmax = 0, 1 #np.nanmin(vals), np.nanmax(vals)

        plt.plot([vmin, vmax], [vmin, vmax], "--", lw=1, color="k", alpha=0.6)

        plt.gca().set_aspect("equal", adjustable="box")
        plt.xlabel("EV (XY)")
        plt.ylabel("EV (HD)")
        plt.ylim(0., 1.)
        plt.xlim(0., 1.)
        plt.title(f"{model_name_plot} / {layer_name_plot}")
        plt.legend(frameon=False, fontsize=8)
        plt.tight_layout()

        scatter_out_path = current_file_path / f"figs/{model_name_plot}_{layer_name_plot}_ev_pitch.pdf"
        scatter_out_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(scatter_out_path)
        plt.show()
        plt.close()

        valid = np.isfinite(ev_xy) & np.isfinite(ev_hd)
        xv, yv = ev_xy[valid], ev_hd[valid]
        xy_dominant = np.sum(xv > yv) / len(xv)
        ratio = yv / (xv + yv + 1e-8)
        print(f"  XY-dominant: {xy_dominant:.1%}, Mean HD/(XY+HD): {np.mean(ratio):.3f}")
        print(f"  Saved scatter to {scatter_out_path}")


## 3D Visualization (using clean activations)

These visualizations use the standard clean activations (not pitch-varied) to show place field structure.

In [ ]:
from src.ann.fig import plot_neuron_2d_3d

def reshape_for_3d_panorama(acts, n_locs, n_hd):
    """
    Reshape panorama activations for 3D plotting.

    Input: (n_locs, n_hd, n_neurons)
    Output: (n_neurons, sqrt(n_locs), sqrt(n_locs), n_hd) assuming square grid
    """
    n_neurons = acts.shape[-1]
    grid_size = int(np.sqrt(n_locs))
    assert grid_size * grid_size == n_locs, f"n_locs={n_locs} is not a perfect square"

    # Reshape: (n_locs, n_hd, n_neurons) -> (grid_x, grid_y, n_hd, n_neurons)
    acts_4d = acts.reshape(grid_size, grid_size, n_hd, n_neurons)

    # Reorder to (n_neurons, grid_x, grid_y, n_hd)
    return np.moveaxis(acts_4d, -1, 0)


# Use clean activations for 3D visualization (0° pitch only, 90 HD)
clean_acts_3d = reshape_for_3d_panorama(clean_acts, n_locs, N_HD_PANORAMA)
print(f"Clean activations for 3D: {clean_acts_3d.shape}")
print(f"  = (n_neurons={n_neurons}, grid_x={N_POS_X}, grid_y={N_POS_Y}, n_hd={N_HD_PANORAMA})")

In [ ]:
# Plot example place-like neurons based on pitch-mode EV results
ev_path = current_file_path / f'results/ann/{model_name}_{layer_name}/ev_all_neurons_pitch.npz'

if ev_path.exists():
    ev_data = np.load(ev_path)
    ev_xy_viz = ev_data['ev_xy']
    ev_hd_viz = ev_data['ev_hd']

    # Find neurons with high XY selectivity (place-like)
    xy_ratio = ev_xy_viz / (ev_xy_viz + ev_hd_viz + 1e-8)
    valid_viz = np.isfinite(xy_ratio)
    top_place_neurons = np.argsort(xy_ratio)[::-1]
    top_place_neurons = [n for n in top_place_neurons if valid_viz[n]][:5]

    print(f"\n=== Top place-like neurons (high XY/total ratio) ===")
    for neuron_idx in top_place_neurons[:3]:
        print(f"Neuron {neuron_idx}: XY ratio = {xy_ratio[neuron_idx]:.3f}")
        fig = plot_neuron_2d_3d(
            clean_acts_3d,
            neuron_idx,
            axes_names=("X", "Y", "HD"),
            overall_title=f"Neuron {neuron_idx} (XY ratio: {xy_ratio[neuron_idx]:.2f})",
            n_slices=4
        )
        fig.write_image(str(current_file_path / f'figs/{model_name}_{layer_name}_neuron_{neuron_idx}_3d_pitch.pdf'))
        fig.show()

    # Also show HD-dominant neurons for comparison
    hd_ratio = ev_hd_viz / (ev_xy_viz + ev_hd_viz + 1e-8)
    top_hd_neurons = np.argsort(hd_ratio)[::-1]
    top_hd_neurons = [n for n in top_hd_neurons if valid_viz[n]][:3]

    print(f"\n=== Top HD-like neurons (high HD/total ratio) ===")
    for neuron_idx in top_hd_neurons[:2]:
        print(f"Neuron {neuron_idx}: HD ratio = {hd_ratio[neuron_idx]:.3f}")
        fig = plot_neuron_2d_3d(
            clean_acts_3d,
            neuron_idx,
            axes_names=("X", "Y", "HD"),
            overall_title=f"Neuron {neuron_idx} (HD ratio: {hd_ratio[neuron_idx]:.2f})",
            n_slices=4
        )
        fig.write_image(str(current_file_path / f'figs/{model_name}_{layer_name}_neuron_{neuron_idx}_hd_3d_pitch.pdf'))
        fig.show()
else:
    print(f"EV data not found at {ev_path}")

## Gabor Filter Analysis

In [ ]:
# Intermediate Layer Analysis - Cell 1: Visualize conv1 filters

from src.ann.fig import visualize_conv1_filters
from src.ann.get_activations import load_model

# Load the predictive coder model (any layer works, we just need the weights)
model_conv1, preprocess_conv1 = load_model(
    'predictive_coder',
    base_path=current_file_path,
    output_layer='conv1',
    device=device
)

# Visualize the 64 learned 7x7 filters
fig = visualize_conv1_filters(model_conv1, ncols=8, normalize=True)
fig.savefig(current_file_path / 'figs/predictive_coder_conv1_filters.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Print filter statistics
weights = model_conv1.core.encoder.conv1.weight.detach().cpu().numpy()
print(f"Conv1 filters shape: {weights.shape}")
print(f"Filter range: [{weights.min():.3f}, {weights.max():.3f}]")

In [ ]:
# Intermediate Layer Analysis - Cell 2: Extract encoder layer activations

encoder_layers = ['conv1', 'down1', 'down2']
encoder_clean_activations = {}
encoder_pitch_activations = {}

for layer in encoder_layers:
    print(f"\n=== Extracting {layer} activations ===")

    model_enc, preprocess_enc = load_model(
        'predictive_coder',
        base_path=current_file_path,
        output_layer=layer,
        device=device
    )

    # Clean activations (for classification and 3D viz)
    clean_acts_path = current_file_path / f'data/ann_activations/predictive_coder_{layer}_panorama_clean/acts.npz'

    if clean_acts_path.exists():
        print(f"  Loading cached clean activations")
        encoder_clean_activations[layer] = np.load(clean_acts_path)['activations']
    else:
        print(f"  Computing clean activations...")
        encoder_clean_activations[layer] = get_panorama_activations(
            model_enc, preprocess_enc, panorama_clean_images,
            model_name='predictive_coder',
            device=device, batch_size=64
        )
        clean_acts_path.parent.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(clean_acts_path, activations=encoder_clean_activations[layer])

    print(f"  Clean shape: {encoder_clean_activations[layer].shape}")

    # Pitch activations (for EV calculation)
    pitch_acts_path = current_file_path / f'data/ann_activations/predictive_coder_{layer}_panorama_pitch/acts.npz'

    if pitch_acts_path.exists():
        print(f"  Loading cached pitch activations")
        encoder_pitch_activations[layer] = np.load(pitch_acts_path)['activations']
    else:
        print(f"  Computing pitch activations...")
        encoder_pitch_activations[layer] = get_panorama_activations(
            model_enc, preprocess_enc, panorama_pitch_images,
            model_name='predictive_coder',
            device=device, batch_size=64
        )
        pitch_acts_path.parent.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(pitch_acts_path, activations=encoder_pitch_activations[layer])

    print(f"  Pitch shape: {encoder_pitch_activations[layer].shape}")

In [ ]:
# Intermediate Layer Analysis - Cell 2: Extract encoder layer activations

encoder_layers = ['conv1', 'down1', 'down2']
encoder_clean_activations = {}
encoder_pitch_activations = {}

for layer in encoder_layers:
    print(f"\n=== Extracting {layer} activations ===")

    model_enc, preprocess_enc = load_model(
        'predictive_coder',
        base_path=current_file_path,
        output_layer=layer,
        device=device
    )

    # Clean activations (for classification and 3D viz)
    clean_acts_path = current_file_path / f'data/ann_activations/predictive_coder_{layer}_panorama_clean/acts.npz'

    if clean_acts_path.exists():
        print(f"  Loading cached clean activations")
        encoder_clean_activations[layer] = np.load(clean_acts_path)['activations']
    else:
        print(f"  Computing clean activations...")
        encoder_clean_activations[layer] = get_panorama_activations(
            model_enc, preprocess_enc, panorama_clean_images,
            model_name='predictive_coder',
            device=device, batch_size=64
        )
        clean_acts_path.parent.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(clean_acts_path, activations=encoder_clean_activations[layer])

    print(f"  Clean shape: {encoder_clean_activations[layer].shape}")

    # Pitch activations (for EV calculation)
    pitch_acts_path = current_file_path / f'data/ann_activations/predictive_coder_{layer}_panorama_pitch/acts.npz'

    if pitch_acts_path.exists():
        print(f"  Loading cached pitch activations")
        encoder_pitch_activations[layer] = np.load(pitch_acts_path)['activations']
    else:
        print(f"  Computing pitch activations...")
        encoder_pitch_activations[layer] = get_panorama_activations(
            model_enc, preprocess_enc, panorama_pitch_images,
            model_name='predictive_coder',
            device=device, batch_size=64
        )
        pitch_acts_path.parent.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(pitch_acts_path, activations=encoder_pitch_activations[layer])

    print(f"  Pitch shape: {encoder_pitch_activations[layer].shape}")

In [ ]:
from src.explained_variance import nan_aware_gaussian_filter, masked_zoom, quantify_variance_corr
from joblib import Parallel, delayed
from tqdm.auto import tqdm

def process_neuron_encoder(n, fold0, fold1, occ, zoom_factor, sigma):
    f0 = nan_aware_gaussian_filter(fold0[:, :, n], sigma=sigma, circular_axes=(1,))
    f1 = nan_aware_gaussian_filter(fold1[:, :, n], sigma=sigma, circular_axes=(1,))
    f0 = masked_zoom(f0, zoom_factor=zoom_factor)
    f1 = masked_zoom(f1, zoom_factor=zoom_factor)
    occ_z = masked_zoom(occ, zoom_factor=zoom_factor)
    ev_xy = 1 - quantify_variance_corr(f0, f1, occ_z, axis=0)[0]
    ev_hd = 1 - quantify_variance_corr(f0, f1, occ_z, axis=1)[0]
    return ev_xy, ev_hd

encoder_ev_results = {}

for layer_name in encoder_layers:
    print(f"\n=== Computing EV for {layer_name} ===")

    pitch_acts = encoder_pitch_activations[layer_name]
    n_locs_enc, n_imgs_enc, n_neurons_enc = pitch_acts.shape
    print(f"  {n_neurons_enc} units")

    pitch_acts_4d = pitch_acts.reshape(n_locs_enc, N_HD_PANORAMA, N_PITCHES, n_neurons_enc)

    fold0_enc, fold1_enc, fold0_occ_enc, fold1_occ_enc = get_splits_deterministic(
        pitch_acts_4d, FOLD0_PITCH_INDICES, FOLD1_PITCH_INDICES
    )
    occ_enc = fold0_occ_enc + fold1_occ_enc
    zoom_factor_enc = (1, N_HD_PANORAMA // N_POS_X)

    sigma = (1, 1)

    results = Parallel(n_jobs=-1)(
        delayed(process_neuron_encoder)(n, fold0_enc, fold1_enc, occ_enc, zoom_factor_enc, sigma)
        for n in tqdm(range(n_neurons_enc), desc=f"  EV ({layer_name})", leave=False)
    )

    ev_xy = np.array([r[0] for r in results])
    ev_hd = np.array([r[1] for r in results])

    encoder_ev_results[layer_name] = (ev_xy, ev_hd)

    ev_out_dir = current_file_path / f'results/ann/predictive_coder_{layer_name}'
    ev_out_dir.mkdir(parents=True, exist_ok=True)
    np.savez(ev_out_dir / 'ev_all_neurons_pitch.npz', ev_xy=ev_xy, ev_hd=ev_hd)
    print(f"  Saved to {ev_out_dir}")


In [ ]:
# Intermediate Layer Analysis - Cell 4: Visualize EV progression across encoder layers

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

print("=== Layer Summary Statistics ===")
for ax, (layer, (ev_xy, ev_hd)) in zip(axes, encoder_ev_results.items()):
    valid = np.isfinite(ev_xy) & np.isfinite(ev_hd) & (ev_xy > 0) & (ev_hd > 0)
    x, y = ev_xy[valid], ev_hd[valid]

    # Subsample for visualization
    n_show = min(500, len(x))
    idx = np.random.choice(len(x), n_show, replace=False)

    ax.scatter(x[idx], y[idx], s=20, alpha=0.5)

    vmin, vmax = 0, 1
    ax.plot([vmin, vmax], [vmin, vmax], '--', color='k', alpha=0.5, label='x=y')

    ax.set_xlim(vmin, vmax)
    ax.set_ylim(vmin, vmax)
    ax.set_xlabel('EV (XY)')
    ax.set_ylabel('EV (HD)')
    ax.set_title(layer)
    ax.set_aspect('equal')

    # Summary stats
    xy_dominant = np.sum(x > y) / len(x)
    denom = x + y
    denom[denom == 0] = 1e-8
    mean_xy_ratio = np.mean(x / denom)

    print(f"{layer:8s}: {xy_dominant:.1%} XY-dominant, mean XY ratio: {mean_xy_ratio:.3f}, n={len(x)}")

plt.tight_layout()
plt.savefig(current_file_path / 'figs/predictive_coder_encoder_ev_scatter_pitch.pdf', bbox_inches='tight')
plt.show()

print("\nInterpretation:")
print("- Early layers (conv1) should show ~50% XY-dominant (no spatial specificity)")
print("- Deeper layers should show increasing XY-dominance (emerging place-like selectivity)")
print("- This progression supports the argument that spatial selectivity emerges from visual features")

In [ ]:
# Intermediate Layer Analysis - Cell 6: Plot conv1 neurons (early layer)

# Pick neurons with highest activation variance (most "interesting" filters)
conv1_var = np.var(conv1_3d, axis=(1, 2, 3))
top_conv1_neurons = np.argsort(conv1_var)[::-1]

print("=== Conv1 Neurons (Early Layer - Visual Features) ===")
print("These should NOT look like place cells or HD cells\n")

for i, neuron_idx in enumerate(top_conv1_neurons[:3]):
    fig = plot_neuron_2d_3d(
        conv1_3d,
        neuron_idx,
        axes_names=("X", "Y", "HD"),
        overall_title=f"Conv1 Neuron {neuron_idx} (Early Visual Feature)",
        n_slices=4
    )
    fig.write_image(str(current_file_path / f'figs/conv1_neuron_{neuron_idx}_3d_pitch.pdf'))
    fig.show()

In [ ]:
# Intermediate Layer Analysis - Cell 7: Plot pre_pred neurons (late layer) for comparison

# Load the EV results to find high-XY neurons (place-like)
ev_path = current_file_path / 'results/ann/predictive_coder_pre_pred/ev_all_neurons_pitch.npz'
if ev_path.exists():
    ev_data = np.load(ev_path)
    ev_xy_prepred = ev_data['ev_xy']
    ev_hd_prepred = ev_data['ev_hd']

    # Find neurons with high XY selectivity (place-like)
    xy_ratio = ev_xy_prepred / (ev_xy_prepred + ev_hd_prepred + 1e-8)
    valid_ratio = np.isfinite(xy_ratio)
    top_place_neurons = np.argsort(xy_ratio)[::-1]
    top_place_neurons = [n for n in top_place_neurons if valid_ratio[n]]

    print("\n=== Pre_pred Neurons (Late Layer - Place-like) ===")
    print("These SHOULD look like place cells with localized firing fields\n")

    for i, neuron_idx in enumerate(top_place_neurons[:3]):
        fig = plot_neuron_2d_3d(
            prepred_3d,
            neuron_idx,
            axes_names=("X", "Y", "HD"),
            overall_title=f"Pre_pred Neuron {neuron_idx} (Place-like, XY ratio: {xy_ratio[neuron_idx]:.2f})",
            n_slices=4
        )
        fig.write_image(str(current_file_path / f'figs/prepred_place_neuron_{neuron_idx}_3d_pitch.pdf'))
        fig.show()

    # Also show HD-dominant neurons
    hd_ratio = ev_hd_prepred / (ev_xy_prepred + ev_hd_prepred + 1e-8)
    top_hd_neurons = np.argsort(hd_ratio)[::-1]
    top_hd_neurons = [n for n in top_hd_neurons if valid_ratio[n]]

    print("\n=== Pre_pred Neurons (Late Layer - HD-like) ===")
    print("These might show HD-like tuning\n")

    for i, neuron_idx in enumerate(top_hd_neurons[:2]):
        fig = plot_neuron_2d_3d(
            prepred_3d,
            neuron_idx,
            axes_names=("X", "Y", "HD"),
            overall_title=f"Pre_pred Neuron {neuron_idx} (HD-like, HD ratio: {hd_ratio[neuron_idx]:.2f})",
            n_slices=4
        )
        fig.write_image(str(current_file_path / f'figs/prepred_hd_neuron_{neuron_idx}_3d_pitch.pdf'))
        fig.show()
else:
    print("EV results for pre_pred not found - run the main analysis first")